# Exercise XP: Agentic AI with RAG (Retrieval-Augmented Generation)

**Goal:** practise agent-like behaviour with open-source frameworks, with **no API keys**:
1. a tiny in-memory **knowledge base** (FAISS vector store, 7 docs);
2. a free **external tool** (Wikipedia search);
3. a **rule-based planner** that decides KB vs. Wikipedia (vs. both);
4. an **answer function** that cites sources (`[kb:doc3]`, `[wiki:Title]`) and admits when evidence is thin;
5. a **quick check** on 3 questions: KB-covered, external, ambiguous.

```
question ─► planner ──► route ∈ {kb, wiki, kb+wiki}
                │
                ├─► KB retriever (FAISS + FakeEmbeddings, topic filter) ─┐
                └─► Wikipedia tool (summaries + titles) ─────────────────┤
                                                                         ▼
                                       answer(): evidence check ─► stub LLM / tiny HF ─► answer + citations
```

In [1]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

In [2]:
import os, re, time, json
os.environ["USE_TF"] = "0"
from importlib.metadata import version
import pandas as pd
pd.set_option("display.max_colwidth", 120)
print({p: version(p) for p in ["langchain", "langchain-community", "langchain-core", "faiss-cpu", "wikipedia"]})

{'langchain': '1.4.4', 'langchain-community': '0.4.2', 'langchain-core': '1.6.9', 'faiss-cpu': '1.15.1', 'wikipedia': '1.4.0'}


## 1. Build the KB retriever
Seven short documents, each with a `source` id (used for citations) and a `topic` tag (used by the planner).

In [3]:
from langchain_core.documents import Document
from langchain_community.embeddings import FakeEmbeddings
from langchain_community.vectorstores import FAISS

KB = [
    ("doc1", "rag", "Retrieval-Augmented Generation (RAG) combines a retriever with a generator: relevant passages are fetched "
                    "from a knowledge source and added to the prompt, so the model answers from evidence instead of memory alone."),
    ("doc2", "rag", "RAG reduces hallucinations and keeps answers up to date without retraining, but its quality depends on "
                    "retrieval: if the right passage is not retrieved, the generator cannot use it."),
    ("doc3", "vector_store", "A vector store such as FAISS indexes embedding vectors and returns the nearest neighbours of a query "
                             "embedding; similarity search is usually cosine or inner product, exact or approximate (HNSW, IVF)."),
    ("doc4", "embeddings", "Embeddings map text to dense vectors so that semantically similar sentences are close in vector space; "
                           "sentence-transformers models like all-MiniLM-L6-v2 produce 384-dimensional embeddings."),
    ("doc5", "agents", "An LLM agent loops over plan, act and observe: it decides which tool to call (search, calculator, database), "
                       "executes it, reads the result and continues until it can answer."),
    ("doc6", "agents", "Planners can be rule-based (if/else routing on keywords) or model-based (the LLM chooses tools via function "
                       "calling); rule-based planners are predictable and cheap but brittle."),
    ("doc7", "langchain", "LangChain provides building blocks for LLM apps: document loaders, text splitters, embeddings, vector stores, "
                          "retrievers, tools and chat models, which can be composed into chains and agents."),
]
docs = [Document(page_content=text, metadata={"source": sid, "topic": topic}) for sid, topic, text in KB]

embeddings = FakeEmbeddings(size=384)           # random vectors: no model download, no key
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"{vectorstore.index.ntotal} docs indexed, dim={vectorstore.index.d}")
for d in retriever.invoke("What is RAG?"):
    print(f"  [kb:{d.metadata['source']}] ({d.metadata['topic']}) {d.page_content[:70]}...")

7 docs indexed, dim=384
  [kb:doc6] (agents) Planners can be rule-based (if/else routing on keywords) or model-base...
  [kb:doc2] (rag) RAG reduces hallucinations and keeps answers up to date without retrai...
  [kb:doc1] (rag) Retrieval-Augmented Generation (RAG) combines a retriever with a gener...


⚠️ `FakeEmbeddings` returns **random vectors**, so the top-3 above are effectively random (the question about RAG may not return doc1 or doc2). It is perfect for testing the plumbing but useless for semantic search.

To keep the agent's retrieval meaningful while still using FakeEmbeddings, the planner identifies the **topic** and we search FAISS with a **metadata filter** on that topic. A one-line swap to real embeddings (`HuggingFaceEmbeddings("sentence-transformers/all-MiniLM-L6-v2")`) would make the similarity itself meaningful.

In [4]:
def kb_search(query, topic=None, k=3):
    kwargs = {"k": k}
    if topic:
        kwargs["filter"] = {"topic": topic}
    return vectorstore.similarity_search(query, **kwargs)

for d in kb_search("What is RAG?", topic="rag"):
    print(f"  [kb:{d.metadata['source']}] {d.page_content[:80]}...")

  [kb:doc2] RAG reduces hallucinations and keeps answers up to date without retraining, but ...
  [kb:doc1] Retrieval-Augmented Generation (RAG) combines a retriever with a generator: rele...


## 2. External tool: Wikipedia (no key)
`WikipediaAPIWrapper` from `langchain_community.utilities.wikipedia` searches Wikipedia and returns page summaries.
- Wikimedia's API policy requires a descriptive **User-Agent**; generic clients are often rate-limited (HTTP 429), so we set one.
- The helper first uses `WikipediaAPIWrapper`. If the MediaWiki search API is rate-limited (shared cloud IPs often get HTTP 429), it falls back to two lighter **official** endpoints that return the same kind of data, live: `opensearch` for titles and the REST page summary. If everything fails, it returns an empty list and the answer function reports missing evidence.

In [5]:
import wikipedia
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

wikipedia.set_user_agent("DI-Bootcamp-RAG-exercise/1.0 (educational notebook; https://github.com/Raphzaf/DI-Bootcamp-Stage2)")
wiki = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=600)
_wiki_cache = {}

import requests
UA = {"User-Agent": "DI-Bootcamp-RAG-exercise/1.0 (educational notebook; https://github.com/Raphzaf/DI-Bootcamp-Stage2)"}

def _wiki_light(query, max_snippets):
    # Lighter official endpoints: opensearch for titles + REST page summaries (also flags disambiguation pages)
    for attempt in range(3):                       # opensearch can be rate-limited too: short retry
        r = requests.get("https://en.wikipedia.org/w/api.php", headers=UA, timeout=20,
                         params={"action": "opensearch", "search": query, "limit": max_snippets, "format": "json"})
        if r.ok:
            break
        time.sleep(10 * (attempt + 1))
    if not r.ok:
        raise RuntimeError(f"opensearch HTTP {r.status_code}")
    titles = r.json()[1]
    out = []
    for t in titles:
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + t.replace(" ", "_"), headers=UA, timeout=20)
        if r.ok:
            j = r.json()
            text = j.get("extract", "")
            if j.get("type") == "disambiguation":
                text = f"{j['title']} may refer to several subjects. " + text
            out.append({"title": j["title"], "snippet": re.sub(r"\s+", " ", text)[:300],
                        "source": "wiki:" + j["title"].replace(" ", "_")})
    return out

_wrapper_ok = True

def wiki_search(query, max_snippets=3):
    """Return [{'title', 'snippet', 'source'}] for a query (empty list if Wikipedia is unreachable)."""
    if query in _wiki_cache:
        return _wiki_cache[query]
    global _wrapper_ok
    out = []
    if _wrapper_ok:
        try:                                            # 1) LangChain WikipediaAPIWrapper (as in the exercise)
            pages = wiki.load(query)
            out = [{"title": p.metadata["title"], "snippet": re.sub(r"\s+", " ", p.page_content)[:300],
                    "source": "wiki:" + p.metadata["title"].replace(" ", "_"), "via": "WikipediaAPIWrapper"} for p in pages[:max_snippets]]
        except Exception as e:
            _wrapper_ok = False                         # circuit breaker: don't hammer a rate-limited endpoint
            print(f"  (WikipediaAPIWrapper failed: {type(e).__name__}, usually an HTTP 429 rate limit -> "
                  "switching to opensearch + REST summaries for this session)")
    if not out:
        try:                                            # 2) lighter live endpoints
            out = [dict(h, via="opensearch+REST") for h in _wiki_light(query, max_snippets)]
        except Exception as e2:
            print(f"  (Wikipedia unreachable: {type(e2).__name__})")
    if out:                                             # never cache failures
        _wiki_cache[query] = out
    return out

for s in wiki_search("Python programming language"):
    print(f"[{s['source']}] {s['snippet'][:150]}...")

  (WikipediaAPIWrapper failed: JSONDecodeError, usually an HTTP 429 rate limit -> switching to opensearch + REST summaries for this session)


[wiki:Python_(programming_language)] Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of signifi...


## 3. Rule-based planner
- Each KB topic has a few **keywords**. The planner counts keyword hits in the question.
- **≥ 2 hits** on a topic → `kb` (strongly covered); **1 hit** → `kb+wiki` (partially covered, so also check Wikipedia); **0 hits** → `wiki`.
- It also extracts a cleaner **search query** for Wikipedia (it strips "what is / who / explain…").

In [6]:
TOPIC_KEYWORDS = {
    "rag": ["rag", "retrieval-augmented", "retrieval augmented", "retrieval", "hallucination", "grounding"],
    "vector_store": ["vector store", "vector database", "faiss", "nearest neighbour", "nearest neighbor", "similarity search", "hnsw"],
    "embeddings": ["embedding", "embeddings", "sentence-transformers", "minilm", "dense vector"],
    "agents": ["agent", "agents", "planner", "tool call", "tools", "plan, act"],
    "langchain": ["langchain", "chain", "retriever", "document loader", "text splitter"],
}
STOP = r"^(what|who|when|where|why|how|explain|tell me about|describe|define)\b( is| are| was| were| does| do| created| invented| founded| wrote| designed| made)?( the| an| a)?\b\s*"

def plan(question):
    q = question.lower()
    scores = {t: sum(kw in q for kw in kws) for t, kws in TOPIC_KEYWORDS.items()}
    topic, best = max(scores.items(), key=lambda kv: kv[1])
    wiki_query = re.sub(STOP, "", question.strip().rstrip("?"), flags=re.I) or question
    if best >= 2:
        return {"route": "kb", "topic": topic, "kb_query": question, "wiki_query": None, "scores": scores}
    if best == 1:
        return {"route": "kb+wiki", "topic": topic, "kb_query": question, "wiki_query": wiki_query, "scores": scores}
    return {"route": "wiki", "topic": None, "kb_query": None, "wiki_query": wiki_query, "scores": scores}

for q in ["How does retrieval-augmented generation reduce hallucination?", "Who created the Python language?", "What is Mercury?"]:
    p = plan(q); print(q, "->", {k: v for k, v in p.items() if k != "scores"})

How does retrieval-augmented generation reduce hallucination? -> {'route': 'kb', 'topic': 'rag', 'kb_query': 'How does retrieval-augmented generation reduce hallucination?', 'wiki_query': None}
Who created the Python language? -> {'route': 'wiki', 'topic': None, 'kb_query': None, 'wiki_query': 'Python language'}
What is Mercury? -> {'route': 'wiki', 'topic': None, 'kb_query': None, 'wiki_query': 'Mercury'}


## 4. Answer function (citations + thin-evidence handling)
1. Execute the plan: KB search (topic-filtered) and/or Wikipedia.
2. **Evidence check:** no evidence → "I don't have enough evidence", plus a suggested follow-up query. Several *unrelated* Wikipedia titles (e.g. Mercury the planet, the element, the god) → the question is **ambiguous**, so we say so and ask which one is meant.
3. Build the prompt (question + numbered context with source ids) and call the LLM:
   - **Default: stub LLM** (`FakeListChatModel`), deterministic and keyless. Its response is an **extractive draft** composed from the top evidence sentences, each followed by its citation. The prompt is still built and sent through the chat-model interface, exactly as with a real model.
   - **Optional:** `USE_HF = True` loads a tiny local Hugging Face pipeline (`sshleifer/tiny-gpt2`). It is only a plumbing test: this 2-layer random-ish model produces gibberish.
4. Always append the list of sources used.

In [7]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

USE_HF = False                          # True -> tiny local HF pipeline (plumbing demo; output is nonsense)
hf_pipe = None
if USE_HF:
    from transformers import pipeline
    hf_pipe = pipeline("text-generation", model="sshleifer/tiny-gpt2")

def first_sentence(text, n=1):
    return " ".join(re.split(r"(?<=[.!?])\s+", text.strip())[:n])

def _distinct_heads(titles):
    return {re.sub(r"\s*\(.*\)", "", t).lower() for t in titles}

def answer(question, verbose=True):
    p = plan(question)
    evidence = []                                   # (source_id, text)
    if p["route"] in ("kb", "kb+wiki"):
        evidence += [("kb:" + d.metadata["source"], d.page_content) for d in kb_search(p["kb_query"], p["topic"], k=3)]
    wiki_hits = wiki_search(p["wiki_query"]) if p["wiki_query"] else []
    evidence += [(h["source"], h["snippet"]) for h in wiki_hits]
    sources = [s for s, _ in evidence]

    # evidence quality
    status = "ok"
    if not evidence:
        status = "no_evidence"
    elif p["route"] == "wiki" and len(wiki_hits) >= 2:
        titles = [h["title"] for h in wiki_hits]
        disambig = any("may refer to" in h["snippet"] or "(disambiguation)" in h["title"] for h in wiki_hits)
        if disambig or len(set(re.sub(r"\s*\(.*\)", "", t) for t in titles)) == 1 and len(set(titles)) > 1:
            status = "ambiguous"                     # e.g. "Mercury (planet)", "Mercury (element)" ...

    context = "\n".join(f"[{s}] {t}" for s, t in evidence)
    prompt = (f"Answer the question using ONLY the context, and cite sources in brackets.\n"
              f"Context:\n{context or '(none)'}\n\nQuestion: {question}\nAnswer:")

    if status == "no_evidence":
        draft = (f"I don't have enough evidence to answer this reliably (no KB match and Wikipedia returned nothing). "
                 f"Suggested follow-up: try a more specific query such as \"{p['wiki_query'] or question} definition\".")
    elif status == "ambiguous":
        options = ", ".join(h["title"] for h in wiki_hits)
        draft = (f"The question is ambiguous: Wikipedia returns several different subjects ({options}). "
                 + " ".join(f"{first_sentence(h['snippet'])} [{h['source']}]" for h in wiki_hits[:2])
                 + f" Suggested follow-up: specify which one you mean, e.g. \"{next((h['title'] for h in wiki_hits if h['title'].lower() != p['wiki_query'].lower()), wiki_hits[0]['title'])}\".")
    else:
        top = evidence[:3] if p["route"] != "kb+wiki" else evidence[:2] + [e for e in evidence if e[0].startswith("wiki:")][:1]
        draft = " ".join(f"{first_sentence(t)} [{s}]" for s, t in top)
        if len(evidence) < 2:
            draft += " (Evidence is thin: only one source found. Consider a follow-up query.)"

    if USE_HF:
        gen = hf_pipe(prompt[-500:], max_new_tokens=30, do_sample=False)[0]["generated_text"][len(prompt[-500:]):]
        final = draft + f"\n\n(tiny-gpt2 continuation, for illustration only: {gen.strip()[:120]!r})"
    else:
        llm = FakeListChatModel(responses=[draft])   # stub chat model: returns the evidence-grounded draft
        final = llm.invoke(prompt).content

    result = {"question": question, "plan": {k: v for k, v in p.items() if k != "scores"}, "keyword_scores": p["scores"],
              "evidence_status": status, "sources": sources, "answer": final + "\n\nSources: " + (", ".join(f"[{s}]" for s in sources) or "none")}
    if verbose:
        print("=" * 100)
        print("QUESTION:", question)
        print("PLAN    :", result["plan"])
        print("SOURCES :", sources, "| evidence:", status,
              ("| wikipedia via " + wiki_hits[0].get("via", "?")) if wiki_hits else "")
        print("ANSWER  :", result["answer"])
    return result

## 5. Quick check: 3 sample questions

In [8]:
questions = [
    "How does retrieval-augmented generation reduce hallucination?",   # KB-covered
    "Who created the Python programming language?",                     # external -> Wikipedia
    "What is Mercury?",                                                 # ambiguous (planet / element / god ...)
]
results = [answer(q) for q in questions]

QUESTION: How does retrieval-augmented generation reduce hallucination?
PLAN    : {'route': 'kb', 'topic': 'rag', 'kb_query': 'How does retrieval-augmented generation reduce hallucination?', 'wiki_query': None}
SOURCES : ['kb:doc2', 'kb:doc1'] | evidence: ok 
ANSWER  : RAG reduces hallucinations and keeps answers up to date without retraining, but its quality depends on retrieval: if the right passage is not retrieved, the generator cannot use it. [kb:doc2] Retrieval-Augmented Generation (RAG) combines a retriever with a generator: relevant passages are fetched from a knowledge source and added to the prompt, so the model answers from evidence instead of memory alone. [kb:doc1]

Sources: [kb:doc2], [kb:doc1]
QUESTION: Who created the Python programming language?
PLAN    : {'route': 'wiki', 'topic': None, 'kb_query': None, 'wiki_query': 'Python programming language'}
SOURCES : ['wiki:Python_(programming_language)'] | evidence: ok | wikipedia via opensearch+REST
ANSWER  : Python is a hig

QUESTION: What is Mercury?
PLAN    : {'route': 'wiki', 'topic': None, 'kb_query': None, 'wiki_query': 'Mercury'}
SOURCES : ['wiki:Mercury', 'wiki:Freddie_Mercury', 'wiki:Mercury_(planet)'] | evidence: ambiguous | wikipedia via opensearch+REST
ANSWER  : The question is ambiguous: Wikipedia returns several different subjects (Mercury, Freddie Mercury, Mercury (planet)). Mercury may refer to several subjects. [wiki:Mercury] Freddie Mercury was a British singer and songwriter who achieved global fame as the lead vocalist and pianist of the rock band Queen. [wiki:Freddie_Mercury] Suggested follow-up: specify which one you mean, e.g. "Freddie Mercury".

Sources: [wiki:Mercury], [wiki:Freddie_Mercury], [wiki:Mercury_(planet)]


In [9]:
pd.DataFrame([{"question": r["question"], "route": r["plan"]["route"], "topic": r["plan"]["topic"],
               "evidence": r["evidence_status"], "sources": ", ".join(r["sources"])} for r in results])

,question,route,topic,evidence,sources
0,How does retrieval-augmented generation reduce hallucination?,kb,rag,ok,"kb:doc2, kb:doc1"
1,Who created the Python programming language?,wiki,NaN,ok,wiki:Python_(programming_language)
2,What is Mercury?,wiki,NaN,ambiguous,"wiki:Mercury, wiki:Freddie_Mercury, wiki:Mercury_(planet)"


In [10]:
# Extra: a partially covered question (1 keyword hit -> KB + Wikipedia) and a nonsense one (no evidence)
_ = answer("What is an intelligent agent?")
_ = answer("Explain the zorbleflax quantum-banana protocol")

  (Wikipedia unreachable: RuntimeError)
QUESTION: What is an intelligent agent?
PLAN    : {'route': 'kb+wiki', 'topic': 'agents', 'kb_query': 'What is an intelligent agent?', 'wiki_query': 'intelligent agent'}
SOURCES : ['kb:doc6', 'kb:doc5'] | evidence: ok 
ANSWER  : Planners can be rule-based (if/else routing on keywords) or model-based (the LLM chooses tools via function calling); rule-based planners are predictable and cheap but brittle. [kb:doc6] An LLM agent loops over plan, act and observe: it decides which tool to call (search, calculator, database), executes it, reads the result and continues until it can answer. [kb:doc5]

Sources: [kb:doc6], [kb:doc5]


QUESTION: Explain the zorbleflax quantum-banana protocol
PLAN    : {'route': 'wiki', 'topic': None, 'kb_query': None, 'wiki_query': 'zorbleflax quantum-banana protocol'}
SOURCES : [] | evidence: no_evidence 
ANSWER  : I don't have enough evidence to answer this reliably (no KB match and Wikipedia returned nothing). Suggested follow-up: try a more specific query such as "zorbleflax quantum-banana protocol definition".

Sources: none


## Quick-check analysis

| Question | Plan | Sources | Outcome |
|---|---|---|---|
| *How does retrieval-augmented generation reduce hallucination?* (KB-covered) | `kb`, topic `rag` (3 keyword hits) | `[kb:doc1]`, `[kb:doc2]` | ✅ Grounded answer from the KB with citations. No external call was needed. |
| *Who created the Python programming language?* (external) | `wiki`, query `"Python programming language"` | `[wiki:Python_(programming_language)]` | ⚠️ Live Wikipedia snippet retrieved and cited, but the answer is **incomplete**: the stub takes the first sentence (what Python *is*), not who created it. The thin-evidence check flagged it ("only one source found"). |
| *What is Mercury?* (ambiguous) | `wiki`, query `"Mercury"` | `[wiki:Mercury]` (disambiguation), `[wiki:Freddie_Mercury]`, `[wiki:Mercury_(planet)]` | ✅ Detected as **ambiguous** (disambiguation page + unrelated titles). The answer says so and suggests a more specific follow-up instead of guessing. |

Extra cases:
- *"What is an intelligent agent?"* is partially covered (1 keyword), so the plan is `kb+wiki`. In this run Wikipedia was rate-limited, so the agent answered from the KB alone (`[kb:doc5]`, `[kb:doc6]`).
- The nonsense question gets no evidence, so it receives an explicit *"I don't have enough evidence"* with a follow-up suggestion, and no fabricated answer.

**What worked / lessons**
- **Routing:** the keyword planner sends KB questions to the vector store and everything else to Wikipedia. The `kb+wiki` middle route covers partial matches. It is cheap and predictable, but brittle: synonyms that are not in the keyword lists will be missed. An LLM or embedding-based router would generalise better.
- **Retrieval:** `FakeEmbeddings` are random, so the *unfiltered* top-3 in Exercise 1 is essentially random. Relevance here comes from the **topic metadata filter**. With real embeddings (e.g. MiniLM), the similarity search itself would be meaningful.
- **Robust tools:** Wikipedia's search API rate-limited this shared cloud IP (HTTP 429). The tool degrades gracefully: it falls back to lighter official endpoints and never caches failures. When nothing works, the answer function reports missing evidence instead of hallucinating.
- **Answer quality:** citations are always attached, and thin or ambiguous evidence is surfaced explicitly. The extractive stub cannot *reason* over the evidence (see the Python question). A real LLM given the same prompt (context with source ids + "cite sources") would extract "created by Guido van Rossum" if that sentence is retrieved. Fetching longer extracts, or the section most relevant to the question, would improve recall.
